In [1]:
import itertools
import os
import random

import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

SEED = 42
TARGET = 0.90
NS = (20, 30, 40)
FRACS = (0.25, 0.5, 1.0)
WEIGHT_ROOT = "./weights"

random.seed(SEED)
np.random.seed(SEED)

## Data

In [2]:
def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def binarise(X):
    """Map each noisy +-1 feature to its sign."""
    return np.where(X > 0, 1.0, -1.0)

In [3]:
def lookup_table_acc(S_train, y_train, S_val, y_val, cols):
    """Val accuracy of a majority-vote lookup table over the bits in `cols`."""
    w = 2 ** np.arange(len(cols))
    idx_tr = ((S_train[:, cols] > 0) * w).sum(1)
    idx_va = ((S_val[:, cols] > 0) * w).sum(1)
    count = np.bincount(idx_tr, minlength=2 ** len(cols))
    pos = np.bincount(idx_tr, weights=y_train, minlength=2 ** len(cols))
    table = (2 * pos > count).astype(int)  # unseen cells default to 0
    return (table[idx_va] == y_val).mean()


def select_features(S_train, y_train, S_val, y_val, min_gain=0.005):
    """Greedy forward selection that may add one bit or a pair of bits per step."""
    n = S_train.shape[1]
    score = lambda cols: lookup_table_acc(S_train, y_train, S_val, y_val, list(cols))
    selected, best = [], 0.5
    while True:
        rest = [j for j in range(n) if j not in selected]
        cands = [[j] for j in rest] + [list(p) for p in itertools.combinations(rest, 2)]
        acc, add = max((score(selected + c), c) for c in cands)
        if acc <= best + min_gain:
            break
        selected, best = selected + add, acc
        print(f"  + {add} -> {sorted(selected)}  val acc {acc:.4f}")
    if not selected:
        print("  no feature set beats chance; keeping all features")
        return list(range(n))
    return sorted(selected)

In [4]:
def tune(name, fit, grid, train, val):
    """Fit `fit(X, y, **params)` for every params in `grid`; keep the best on val (first wins ties)."""
    (Xtr, ytr), (Xva, yva) = train, val
    best = None
    for params in grid:
        model = fit(Xtr, ytr, **params)
        acc = model.score(Xva, yva)
        if best is None or acc > best[0]:
            best = (acc, params, model)
    print(f"  {name}: best {best[1]}  val acc {best[0]:.4f}")
    return best


def fit_logreg(X, y, C):
    """Logistic regression on the sign bits."""
    return LogisticRegression(C=C, max_iter=5000, random_state=SEED).fit(X, y)


def fit_rbf_logreg(X, y, gamma, n_components, C):
    """Nystroem RBF features with landmarks drawn from the distinct inputs of X, then logistic regression."""
    landmarks = np.unique(X, axis=0)
    rbf = Nystroem(kernel="rbf", gamma=gamma, n_components=min(n_components, len(landmarks)),
                   random_state=SEED).fit(landmarks)
    logreg = LogisticRegression(C=C, max_iter=5000, random_state=SEED).fit(rbf.transform(X), y)
    return make_pipeline(rbf, logreg)


logreg_grid = [dict(C=C) for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]]
rbf_grid = [dict(gamma=g, n_components=k, C=C)
            for g in [0.25, 0.5, 1.0] for k in [256, 1024] for C in [1.0, 10.0, 100.0]]

# (display name, weight folder, fit function, grid)
model_specs = [
    ("Logistic regression", "logistic_regression", fit_logreg, logreg_grid),
    ("RBF features + LR", "rbf_logistic_regression", fit_rbf_logreg, rbf_grid),
]
W = max(len(name) for name, *_ in model_specs)

## All datasets and scales

In [5]:
def run(n, frac):
    """Feature selection on the first `frac` of train/val, tune every model, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}: {m} train / {mv} val / {len(y_te)} test ===")

    cols = select_features(S_tr, y_tr, S_va, y_va)
    print(f"  selected {len(cols)} features: {cols}")
    F_tr, F_va, F_te = S_tr[:, cols], S_va[:, cols], S_te[:, cols]

    rows = []
    for model_name, folder, fit, grid in model_specs:
        val_acc, params, model = tune(model_name, fit, grid, (F_tr, y_tr), (F_va, y_va))
        test_acc = model.score(F_te, y_te)
        out_dir = os.path.join(WEIGHT_ROOT, folder, f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.joblib")
        joblib.dump({"model": model, "features": cols, "params": params}, path)
        print(f"  {model_name:<{W}s} val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": model_name, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results = [row for n in NS for frac in FRACS for row in run(n, frac)]


=== n=20, 25%: 1800 train / 600 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5550
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6117
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6583
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7350
  + [0, 2] -> [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]  val acc 0.8700
  selected 10 features: [0, 1, 2, 4, 5, 7, 9, 10, 12, 15]
  Logistic regression: best {'C': 0.001}  val acc 0.5250
  Logistic regression val 0.5250  test 0.5025  -> ./weights\logistic_regression\n20\25pct\model.joblib
  RBF features + LR: best {'gamma': 1.0, 'n_components': 1024, 'C': 1.0}  val acc 0.8483
  RBF features + LR   val 0.8483  test 0.8583  -> ./weights\rbf_logistic_regression\n20\25pct\model.joblib

=== n=20, 50%: 3600 train / 1200 val / 2400 test ===
  + [4, 9] -> [4, 9]  val acc 0.5750
  + [10, 12] -> [4, 9, 10, 12]  val acc 0.6283
  + [5, 15] -> [4, 5, 9, 10, 12, 15]  val acc 0.6842
  + [1, 7] -> [1, 4, 5, 7, 9, 10, 12, 15]  val acc 0.7692
  + [0, 2] -> [0, 1, 2, 4

## Summary

In [6]:
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'model':<{W}s} | {'#feat':>5} | {'val':>6} | {'test':>6}")
for r in results:
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{r['n']:>3} | {r['frac']:>6.0%} | {r['train_size']:>6} | {r['model']:<{W}s} | "
          f"{len(r['features']):>5} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | model               | #feat |    val |   test
 20 |    25% |   1800 | Logistic regression |    10 | 0.5250 | 0.5025  < target
 20 |    25% |   1800 | RBF features + LR   |    10 | 0.8483 | 0.8583  < target
 20 |    50% |   3600 | Logistic regression |    10 | 0.4942 | 0.5062  < target
 20 |    50% |   3600 | RBF features + LR   |    10 | 0.9342 | 0.9354
 20 |   100% |   7200 | Logistic regression |    10 | 0.5279 | 0.5238  < target
 20 |   100% |   7200 | RBF features + LR   |    10 | 0.9533 | 0.9592
 30 |    25% |   3000 | Logistic regression |    10 | 0.4660 | 0.4903  < target
 30 |    25% |   3000 | RBF features + LR   |    10 | 0.9070 | 0.9058
 30 |    50% |   6000 | Logistic regression |    10 | 0.4790 | 0.5028  < target
 30 |    50% |   6000 | RBF features + LR   |    10 | 0.9545 | 0.9523
 30 |   100% |  12000 | Logistic regression |    10 | 0.4968 | 0.5240  < target
 30 |   100% |  12000 | RBF features + LR   |    10 | 0.9565 | 0.9580
 40 |    25% |   480

## Feature-selection ablation

In [7]:
print(f"{'n':>3} | {'model':<{W}s} | {'selected val':>12} | {'selected test':>13} | {'all val':>7} | {'all test':>8}")
for r in results:
    if r["frac"] != 1.0:
        continue
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    fit = next(f for name, _, f, _ in model_specs if name == r["model"])
    model = fit(binarise(X_train), y_train, **r["params"])
    print(f"{r['n']:>3} | {r['model']:<{W}s} | {r['val']:>12.4f} | {r['test']:>13.4f} | "
          f"{model.score(binarise(X_val), y_val):>7.4f} | {model.score(binarise(X_test), y_test):>8.4f}")

  n | model               | selected val | selected test | all val | all test
 20 | Logistic regression |       0.5279 |        0.5238 |  0.5000 |   0.5025
 20 | RBF features + LR   |       0.9533 |        0.9592 |  0.5300 |   0.5367
 30 | Logistic regression |       0.4968 |        0.5240 |  0.4883 |   0.5052
 30 | RBF features + LR   |       0.9565 |        0.9580 |  0.4870 |   0.5072
 40 | Logistic regression |       0.5044 |        0.5184 |  0.5041 |   0.5052
 40 | RBF features + LR   |       0.9569 |        0.9631 |  0.5053 |   0.4961


Reload every saved model and check it reproduces its test accuracy.

In [8]:
for r in results:
    saved = joblib.load(r["path"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    acc = saved["model"].score(binarise(X_test)[:, saved["features"]], y_test)
    assert abs(acc - r["test"]) < 1e-9, (r["path"], acc, r["test"])
    print(f"{r['path']} -> test {acc:.4f}")

./weights\logistic_regression\n20\25pct\model.joblib -> test 0.5025
./weights\rbf_logistic_regression\n20\25pct\model.joblib -> test 0.8583
./weights\logistic_regression\n20\50pct\model.joblib -> test 0.5062
./weights\rbf_logistic_regression\n20\50pct\model.joblib -> test 0.9354
./weights\logistic_regression\n20\100pct\model.joblib -> test 0.5238
./weights\rbf_logistic_regression\n20\100pct\model.joblib -> test 0.9592
./weights\logistic_regression\n30\25pct\model.joblib -> test 0.4903
./weights\rbf_logistic_regression\n30\25pct\model.joblib -> test 0.9058
./weights\logistic_regression\n30\50pct\model.joblib -> test 0.5028
./weights\rbf_logistic_regression\n30\50pct\model.joblib -> test 0.9523
./weights\logistic_regression\n30\100pct\model.joblib -> test 0.5240
./weights\rbf_logistic_regression\n30\100pct\model.joblib -> test 0.9580
./weights\logistic_regression\n40\25pct\model.joblib -> test 0.5009
./weights\rbf_logistic_regression\n40\25pct\model.joblib -> test 0.9508
./weights\logist

## No feature selection (ablation)

The same models, grids, split and data fractions as above, but every model is tuned on all n sign features instead of the selected ones. The best model per (n, fraction) is saved to `weights/<model>_no_fs/n<n>/<pct>pct/model.joblib` in the same layout, with `features` set to all n columns.

In [9]:
def run_no_fs(n, frac):
    """Use the first `frac` of train/val with all n features, tune every model, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}, no FS: {m} train / {mv} val / {len(y_te)} test ===")

    cols = list(range(n))  # no feature selection: all features are used
    width = max(len(name) for name, *_ in model_specs)
    rows = []
    for model_name, folder, make, grid in model_specs:
        val_acc, params, model = tune(model_name, make, grid, (S_tr, y_tr), (S_va, y_va))
        test_acc = model.score(S_te, y_te)
        out_dir = os.path.join(WEIGHT_ROOT, f"{folder}_no_fs", f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.joblib")
        joblib.dump({"model": model, "features": cols, "params": params}, path)
        print(f"  {model_name:<{width}s} val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": model_name, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results_no_fs = [row for n in NS for frac in FRACS for row in run_no_fs(n, frac)]


=== n=20, 25%, no FS: 1800 train / 600 val / 2400 test ===
  Logistic regression: best {'C': 0.001}  val acc 0.5100
  Logistic regression val 0.5100  test 0.5088  -> ./weights\logistic_regression_no_fs\n20\25pct\model.joblib
  RBF features + LR: best {'gamma': 0.5, 'n_components': 1024, 'C': 100.0}  val acc 0.5517
  RBF features + LR   val 0.5517  test 0.5233  -> ./weights\rbf_logistic_regression_no_fs\n20\25pct\model.joblib

=== n=20, 50%, no FS: 3600 train / 1200 val / 2400 test ===
  Logistic regression: best {'C': 0.001}  val acc 0.4975
  Logistic regression val 0.4975  test 0.5167  -> ./weights\logistic_regression_no_fs\n20\50pct\model.joblib
  RBF features + LR: best {'gamma': 0.25, 'n_components': 256, 'C': 100.0}  val acc 0.5442
  RBF features + LR   val 0.5442  test 0.5196  -> ./weights\rbf_logistic_regression_no_fs\n20\50pct\model.joblib

=== n=20, 100%, no FS: 7200 train / 2400 val / 2400 test ===
  Logistic regression: best {'C': 0.001}  val acc 0.5000
  Logistic regressio

### Summary without feature selection

In [10]:
width = max(len(name) for name, *_ in model_specs)
print(f"{'n':>3} | {'subset':>6} | {'train':>6} | {'model':<{width}s} | {'#feat':>5} | {'val':>6} | {'test':>6}")
for r in results_no_fs:
    flag = "" if r["test"] >= TARGET else "  < target"
    print(f"{r['n']:>3} | {r['frac']:>6.0%} | {r['train_size']:>6} | {r['model']:<{width}s} | "
          f"{len(r['features']):>5} | {r['val']:.4f} | {r['test']:.4f}{flag}")

  n | subset |  train | model               | #feat |    val |   test
 20 |    25% |   1800 | Logistic regression |    20 | 0.5100 | 0.5088  < target
 20 |    25% |   1800 | RBF features + LR   |    20 | 0.5517 | 0.5233  < target
 20 |    50% |   3600 | Logistic regression |    20 | 0.4975 | 0.5167  < target
 20 |    50% |   3600 | RBF features + LR   |    20 | 0.5442 | 0.5196  < target
 20 |   100% |   7200 | Logistic regression |    20 | 0.5000 | 0.5025  < target
 20 |   100% |   7200 | RBF features + LR   |    20 | 0.5446 | 0.5375  < target
 30 |    25% |   3000 | Logistic regression |    30 | 0.4860 | 0.5025  < target
 30 |    25% |   3000 | RBF features + LR   |    30 | 0.4850 | 0.5072  < target
 30 |    50% |   6000 | Logistic regression |    30 | 0.4815 | 0.5105  < target
 30 |    50% |   6000 | RBF features + LR   |    30 | 0.4760 | 0.5042  < target
 30 |   100% |  12000 | Logistic regression |    30 | 0.4893 | 0.5052  < target
 30 |   100% |  12000 | RBF features + LR   |    3

Reload every saved no-FS model and check it reproduces its test accuracy.

In [11]:
for r in results_no_fs:
    saved = joblib.load(r["path"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    acc = saved["model"].score(binarise(X_test)[:, saved["features"]], y_test)
    assert abs(acc - r["test"]) < 1e-9, (r["path"], acc, r["test"])
    print(f"{r['path']} -> test {acc:.4f}")

./weights\logistic_regression_no_fs\n20\25pct\model.joblib -> test 0.5088
./weights\rbf_logistic_regression_no_fs\n20\25pct\model.joblib -> test 0.5233
./weights\logistic_regression_no_fs\n20\50pct\model.joblib -> test 0.5167
./weights\rbf_logistic_regression_no_fs\n20\50pct\model.joblib -> test 0.5196
./weights\logistic_regression_no_fs\n20\100pct\model.joblib -> test 0.5025
./weights\rbf_logistic_regression_no_fs\n20\100pct\model.joblib -> test 0.5375
./weights\logistic_regression_no_fs\n30\25pct\model.joblib -> test 0.5025
./weights\rbf_logistic_regression_no_fs\n30\25pct\model.joblib -> test 0.5072
./weights\logistic_regression_no_fs\n30\50pct\model.joblib -> test 0.5105
./weights\rbf_logistic_regression_no_fs\n30\50pct\model.joblib -> test 0.5042
./weights\logistic_regression_no_fs\n30\100pct\model.joblib -> test 0.5052
./weights\rbf_logistic_regression_no_fs\n30\100pct\model.joblib -> test 0.5072
./weights\logistic_regression_no_fs\n40\25pct\model.joblib -> test 0.4952
./weights\